In [ ]:
# Export a NEW CSV with English `title`/`description` and emojis removed
# If needed, run once: %pip install deep-translator

import re
from pathlib import Path
import pandas as pd

try:
    from deep_translator import GoogleTranslator
except ImportError as e:
    raise ImportError("Install first with: %pip install deep-translator") from e

DATA_PATH = Path("data/tayara.csv")
OUTPUT_CSV = Path("data/tayara_en_noemoji.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {DATA_PATH.resolve()}")

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip"
)

EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002700-\U000027BF"  # dingbats
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "]+",
    flags=re.UNICODE,
)


def remove_emojis(text):
    if pd.isna(text):
        return ""
    return EMOJI_RE.sub("", str(text)).strip()


def translate_series_to_en(series, batch_size=30):
    s = series.fillna("").astype(str).str.strip()
    unique_texts = [x for x in s.unique().tolist() if x]
    translator = GoogleTranslator(source="auto", target="en")
    cache = {}

    for i in range(0, len(unique_texts), batch_size):
        batch = unique_texts[i:i + batch_size]
        try:
            translated = translator.translate_batch(batch)
            if isinstance(translated, str):
                translated = [translated]
            if translated is None:
                translated = batch

            for src, dst in zip(batch, translated):
                clean_dst = remove_emojis(dst) if isinstance(dst, str) else ""
                cache[src] = clean_dst if clean_dst else src

            if len(translated) < len(batch):
                for src in batch[len(translated):]:
                    cache[src] = src
        except Exception:
            for src in batch:
                try:
                    dst = translator.translate(src)
                    clean_dst = remove_emojis(dst) if isinstance(dst, str) else ""
                    cache[src] = clean_dst if clean_dst else src
                except Exception:
                    cache[src] = src

    return s.map(lambda x: cache.get(x, x))


translated_df = df.copy()

# Ensure columns exist
for col in ["title", "description"]:
    if col not in translated_df.columns:
        translated_df[col] = ""

# Keep originals
translated_df["title_original"] = translated_df["title"]
translated_df["description_original"] = translated_df["description"]

# Remove emojis before translation
translated_df["title_clean"] = translated_df["title_original"].map(remove_emojis)
translated_df["description_clean"] = translated_df["description_original"].map(remove_emojis)

# Translate to English
translated_df["title_en"] = translate_series_to_en(translated_df["title_clean"])
translated_df["description_en"] = translate_series_to_en(translated_df["description_clean"])

# Replace model text columns with English versions
translated_df["title"] = translated_df["title_en"]
translated_df["description"] = translated_df["description_en"]

translated_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Saved translated file to: {OUTPUT_CSV.resolve()}")
display(translated_df[["title_original", "title", "description_original", "description"]].head(3))